# Exercício 06 — RAG Jurídico

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## 1. Documentos fictícios e chunking simples

In [ ]:
DOCS = [
    {"id": "contrato_servicos.txt", "texto": "Cláusula 7: prazo de resposta a pedidos de suporte é de 5 dias úteis."},
    {"id": "politica_privacidade.txt", "texto": "Os dados são conservados por 24 meses após termo do contrato."},
]
CHUNK_SIZE = 220

def chunk(doc: dict) -> list[dict]:
    t = doc["texto"]
    out = []
    for i in range(0, len(t), CHUNK_SIZE):
        out.append({"id": doc["id"], "chunk": t[i : i + CHUNK_SIZE]})
    return out

chunks = [c for d in DOCS for c in chunk(d)]
len(chunks), chunks[:2]


## 2. Embeddings Gemini + Chroma (em memória)

**Nota:** o aviso `onnxruntime … Unknown CPU vendor` é habitual em algumas VMs e pode ignorar-se. O modelo **`models/text-embedding-004`** já não está disponível na Gemini API — usa-se **`gemini-embedding-001`** (ou `GEMINI_EMBEDDING_MODEL` no `.env`).


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os

# `text-embedding-004` já não existe em embedContent; valores antigos no .env são ignorados aqui.
_raw = (
    os.environ.get("GEMINI_EMBEDDING_MODEL_EX06")
    or os.environ.get("GEMINI_EMBEDDING_MODEL")
    or "gemini-embedding-001"
).replace("models/", "")
_embed = "gemini-embedding-001" if "text-embedding-004" in _raw else (_raw or "gemini-embedding-001")
emb = GoogleGenerativeAIEmbeddings(model=_embed)
vectordb = Chroma.from_documents(
    [Document(page_content=c["chunk"], metadata={"fonte": c["id"]}) for c in chunks],
    embedding=emb,
)
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

pergunta = "Qual é o prazo de resposta a pedidos de suporte?"
docs = retriever.invoke(pergunta)
docs


## 3. Geração com contexto

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.2,
)
ctx = "\n".join(f"[{d.metadata.get('fonte')}] {d.page_content}" for d in docs)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde só com base no contexto; se faltar info, diz-o. Português europeu."),
    ("human", "Contexto:\n{ctx}\n\nPergunta: {q}"),
])
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"ctx": ctx, "q": pergunta}))


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
